# M1.S2 - Evolution of supercomputing and the modern HPC landscape
## Why supercomputers changed

The central idea of this session is simple:

> **HPC evolves when the bottleneck moves.**

Early systems improved mainly through faster electronics and smarter processors. Over time, limits in power, heat, memory bandwidth, communication and data movement made that approach insufficient.

The result was a sequence of architectural shifts:

```text
faster processors
      ->
vector / SIMD
      ->
many processors
      ->
commodity clusters
      ->
multicore + GPUs
      ->
heterogeneous exascale systems
```

### What you will practice

By the end of the notebook you should be able to:

1. reconstruct the broad evolution of supercomputing;
2. connect each major architectural shift to the bottleneck that motivated it;
3. explain why parallelism became necessary;
4. distinguish vector/SIMD, distributed processing and heterogeneous computing;
5. explain why software had to evolve with the hardware;
6. interpret FLOPS, HPL and time-to-solution correctly;
7. compare modern HPC systems by more than peak performance;
8. inspect how the SciTech cluster reflects this evolution.

Use the same cycle throughout:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

## 1 - Rebuild the evolution

Put these eras in chronological order before running the next cell:

- Multicore + GPU
- Stored-program electronic computing
- Commodity clusters / MPP
- Vector processing
- Mechanical / electromechanical computing
- SIMD arrays
- Exascale + AI
- Instruction-level parallelism

### Your prediction

Write your order here before running the checker.

In [ ]:
eras = [
    "Mechanical / electromechanical computing",
    "Stored-program electronic computing",
    "Instruction-level parallelism",
    "Vector processing",
    "SIMD arrays",
    "Commodity clusters / MPP",
    "Multicore + GPU",
    "Exascale + AI",
]

for i, era in enumerate(eras, start=1):
    print(f"{i}. {era}")

### Observe

The important pattern is not the exact date of every machine. It is the direction of travel:

- more work overlapped inside one processor;
- then one instruction acted on many data elements;
- then many processors worked together;
- then standard nodes were connected into clusters;
- then multicore CPUs and GPUs became common;
- now systems combine different processing, memory and interconnect technologies.

### Explain

What drove these changes?

<details>
<summary><strong>Show explanation</strong></summary>

Each generation solved an important limit and exposed another.

Examples:

- faster electronics increased speed, but power and heat became harder;
- vector processors improved arithmetic throughput, but memory bandwidth mattered more;
- many processors increased total compute, but communication and synchronization became important;
- GPUs increased throughput, but data movement and programmability became critical;
- exascale systems must balance compute, memory, communication and energy.

</details>

## 2 - From scalar work to vector-style work

Vector supercomputers were built around a powerful idea:

> **Apply the same operation efficiently to many data elements.**

That idea still appears today in CPU SIMD instructions and GPU programming.

We can demonstrate it with a simple array operation.

In [ ]:
import time
import numpy as np

N = 2_000_000
a = np.arange(N, dtype=np.float64)
b = np.arange(N, dtype=np.float64) * 0.5

print("Elements:", N)
print("Data per array:", f"{a.nbytes / 1024**2:.1f} MiB")

### Predict

Both implementations compute:

```text
c[i] = 2 * a[i] + b[i]
```

Which do you expect to be faster?

1. a Python loop that processes one element at a time;
2. NumPy, which delegates the array operation to optimized compiled code.

This is not a benchmark of historical vector machines. It is a small demonstration of the **same-operation-many-data** idea.

In [ ]:
def scalar_python(a, b):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = 2.0 * a[i] + b[i]
    return out

t0 = time.perf_counter()
c_scalar = scalar_python(a, b)
t_scalar = time.perf_counter() - t0

t0 = time.perf_counter()
c_vector = 2.0 * a + b
t_vector = time.perf_counter() - t0

print(f"Python element-by-element: {t_scalar:.4f} s")
print(f"Array operation:           {t_vector:.4f} s")
print(f"Observed ratio:            {t_scalar/t_vector:.1f}x")
print("Same result:", np.allclose(c_scalar, c_vector))

### Explain

The main lesson is not the exact speedup.

The useful idea is:

> **Regular data-parallel work can be executed much more efficiently when the hardware and software operate on groups of data rather than treating every element as a separate high-level operation.**

Modern CPUs use SIMD/vector instructions, and GPUs extend the same broad idea to very large numbers of parallel operations.

## 3 - Why did "just increase the clock" stop working?

Imagine a processor at 3 GHz. Applications still need more performance.

A naive idea is to keep increasing frequency:

```text
3 GHz -> 4 GHz -> 5 GHz -> 10 GHz
```

But higher switching rates increase power consumption and heat.

At the same time, processors can often execute operations faster than data can be supplied from memory.

These are two important historical limits:

- **power wall**
- **memory wall**

### Quick check

Match each statement to the correct wall.

**A.** More switching produces unacceptable energy use and heat.  
**B.** Arithmetic capability improves faster than data can be supplied from memory.

<details>
<summary><strong>Show explanation</strong></summary>

- **A = power wall**
- **B = memory wall**

The response was not one single invention. Computing shifted toward multicore processors, GPUs, wider memory systems, better locality and more explicit parallelism.

</details>

## 4 - A simple memory-wall experiment

We can make the memory problem visible with two NumPy operations.

The first streams through a large array and performs very little arithmetic.

The second reuses a small array many times, keeping the working set easier to cache.

The exact timing depends on the machine. Focus on the pattern.

In [ ]:
large = np.ones(20_000_000, dtype=np.float64)
small = np.ones(32_000, dtype=np.float64)

# Stream through a large array once
t0 = time.perf_counter()
stream_result = large * 1.000001
t_stream = time.perf_counter() - t0

# Reuse a much smaller working set repeatedly
work = small.copy()
t0 = time.perf_counter()
for _ in range(200):
    work = work * 1.000001 + 0.000001
t_reuse = time.perf_counter() - t0

print(f"Large streaming pass:    {t_stream:.4f} s")
print(f"Small reused working set:{t_reuse:.4f} s")
print()
print("Large array size:", f"{large.nbytes / 1024**2:.1f} MiB")
print("Small array size:", f"{small.nbytes / 1024:.1f} KiB")

### Explain

Do not compare the two times as if they perform the same amount of work. They do not.

Instead, notice the architectural question:

> **How much data must move between memory and the processor for each useful operation?**

As arithmetic became cheaper and faster, the cost of moving data became increasingly important.

This is why modern HPC pays so much attention to:

- cache hierarchy;
- memory bandwidth;
- high-bandwidth memory;
- locality;
- CPU-GPU data movement;
- network communication;
- storage I/O.

## 5 - From one processor to many processors

When one processor could no longer deliver enough performance, systems increasingly used many processors.

That changed the programming problem.

With one shared processor, data is local.

With many distributed processors:

```text
processor 0  <---- network ---->  processor 1
    data A                         data B
```

the program must coordinate work and exchange data.

This is why message passing became central to distributed-memory HPC.

### Predict

What new costs appear when a problem is split across many processors or nodes?

<details>
<summary><strong>Show explanation</strong></summary>

Typical costs include:

- communication;
- synchronization;
- data distribution;
- load imbalance;
- network latency and bandwidth limits.

More processors increase available compute, but they also create new coordination costs.

</details>

## 6 - Why commodity clusters changed HPC

A major shift occurred when large HPC systems could be assembled from more standard components:

```text
standard processors
      +
network
      +
Linux / system software
      +
parallel programming libraries
      =
commodity cluster
```

This made systems more modular and economically scalable than relying only on fully proprietary supercomputer designs.

### Classification

Which statements describe the cluster model?

1. Standard processors can be combined into larger systems.
2. Communication is no longer necessary.
3. Distributed-memory software becomes important.
4. Systems can grow by adding nodes.
5. Parallel programming is eliminated.

<details>
<summary><strong>Show explanation</strong></summary>

Correct:

- **1**
- **3**
- **4**

Incorrect:

- **2** - communication becomes more important, not less.
- **5** - clusters require parallel programming to exploit many nodes effectively.

</details>

## 7 - Hardware changed, so software changed too

Architecture and programming models evolved together.

A useful simplified mapping is:

| Architectural shift | Typical software response |
|---|---|
| Stored-program computers | machine code, early compiled languages |
| Vector processors | vectorization |
| Distributed-memory systems | message passing / MPI |
| Multicore CPUs | threads / OpenMP |
| GPUs | CUDA, OpenACC, accelerator libraries |
| Heterogeneous AI/HPC systems | MPI + accelerator models + AI frameworks |

### Predict

A sequential CPU program is moved unchanged to a CPU+GPU supercomputer.

Should you expect a large speedup automatically?

<details>
<summary><strong>Show explanation</strong></summary>

No.

Hardware acceleration helps only if the algorithm and software expose enough suitable parallel work and manage data movement effectively.

A sequential program can remain sequential on a much more powerful machine.

</details>

## 8 - Performance metrics: FLOPS are useful, but not enough

**FLOPS** means floating-point operations per second.

Prefixes describe scale:

```text
MFLOP/s  = 10^6 FLOP/s
GFLOP/s  = 10^9 FLOP/s
TFLOP/s  = 10^12 FLOP/s
PFLOP/s  = 10^15 FLOP/s
EFLOP/s  = 10^18 FLOP/s
```

Peak FLOPS describe one aspect of a system.

For an application, a more practical question is often:

> **How long does my problem take to complete?**

### Scenario

System A reaches 1.5 EFLOP/s on a dense linear algebra benchmark.

System B reaches 1.0 EFLOP/s, but has much higher memory bandwidth.

Your application repeatedly streams very large arrays from memory.

Which system is necessarily faster?

<details>
<summary><strong>Show explanation</strong></summary>

Neither can be declared the winner from HPL performance alone.

The memory-intensive application may perform better on System B if memory bandwidth is its real bottleneck.

A benchmark measures a particular behavior. Application performance depends on how well the complete system matches the workload.

</details>

## 9 - HPL, HPCG and time-to-solution

The TOP500 ranking uses **HPL**, a dense linear algebra benchmark.

HPL is important because it provides a consistent way to compare large systems.

But it is not the only useful perspective.

Different workloads stress different parts of a machine:

- dense arithmetic;
- memory movement;
- sparse computation;
- communication;
- storage;
- energy efficiency.

### Three measurements

For each metric, identify what it tells you best.

| Metric | Main question |
|---|---|
| HPL | ? |
| Application runtime | ? |
| Power / energy | ? |

<details>
<summary><strong>Show explanation</strong></summary>

- **HPL:** how well the system performs a highly parallel dense linear algebra workload.
- **Application runtime:** how quickly the workload you actually care about finishes.
- **Power / energy:** how much electrical/energy cost is associated with the delivered performance.

No single metric describes every workload.

</details>

## 10 - Modern HPC is an ecosystem

A modern HPC system is not just "a lot of CPUs".

It can combine:

- CPU clusters;
- GPUs and AI accelerators;
- high-bandwidth memory;
- fast interconnects;
- parallel storage;
- cloud resources;
- specialized accelerators;
- compilers, libraries and schedulers.

### Workload matching

For each workload, choose the resource that seems especially important.

**A.** Train a large transformer model.  
**B.** Run millions of independent small parameter studies.  
**C.** Stream and analyze petabytes of scientific data.  
**D.** Tightly coupled multi-node simulation.  
**E.** Domain-specific low-latency processing.

<details>
<summary><strong>Show explanation</strong></summary>

Possible answers:

- **A:** GPUs/AI accelerators + high-bandwidth memory + fast interconnect.
- **B:** many CPU/GPU nodes and good scheduling throughput.
- **C:** storage bandwidth + memory + distributed data processing.
- **D:** CPUs/GPUs + very low-latency, high-bandwidth interconnect.
- **E:** specialized accelerators such as FPGA may be relevant.

There is rarely one universally best machine. The architecture should match the workload.

</details>

## 11 - Compare four modern system designs

The following simplified profiles illustrate why exascale-class systems can look very different.

| System | Main architecture clue | Main lesson |
|---|---|---|
| LineShine | highly parallel custom CPU-only design | GPUs are not the only route to extreme scale |
| El Capitan | AMD MI300A CPU+GPU APUs with high-bandwidth unified memory | bring compute and memory closer together |
| Aurora | CPUs + large numbers of GPUs + very large storage system | compute, memory, storage and interconnect all matter |
| JUPITER Booster | GH200 CPU+GPU nodes with high-bandwidth memory | tightly coupled heterogeneous nodes for simulation and AI |

### Challenge

Match the design clue to the architectural pressure.

1. Unified / high-bandwidth memory
2. Many GPUs
3. Very large parallel storage
4. CPU-only massively parallel design

Possible pressures:

- arithmetic throughput;
- data movement;
- data volume / I/O;
- scalable conventional CPU parallelism.

<details>
<summary><strong>Show explanation</strong></summary>

A reasonable mapping is:

- **Unified / high-bandwidth memory -> data movement**
- **Many GPUs -> arithmetic throughput / massive parallelism**
- **Very large parallel storage -> data volume / I/O**
- **CPU-only massively parallel design -> scalable conventional CPU parallelism**

The broader lesson is that exascale describes a performance scale, not one mandatory architecture.

</details>

## 12 - Read the SciTech cluster as an evolutionary artifact

The SciTech environment is a small teaching cluster, but it still reflects several ideas from the history of HPC.

We will inspect:

- CPU topology;
- multiple nodes;
- CPU and GPU partitions;
- the scheduler;
- software modules.

No heavy job is submitted in this session.

In [ ]:
import os
import subprocess

def run_command(command):
    print("$", command)
    p = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()

print("Cluster inspection only.")

In [ ]:
run_command("hostname")
run_command("lscpu | grep -E 'Architecture|CPU\\(s\\)|Socket|Core|Thread|Model name' | head -n 12")

In [ ]:
if subprocess.run("command -v sinfo >/dev/null 2>&1", shell=True, executable="/bin/bash").returncode == 0:
    run_command("sinfo -a -o '%P %a %l %D %G'")
    run_command("sinfo -a -N -o '%N %P %t %G'")
else:
    print("Slurm is not available in this environment.")

In [ ]:
print("Current allocation clues:")
for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_PARTITION",
    "SLURM_NODELIST",
    "SLURM_CPUS_PER_TASK",
]:
    print(f"{key}={os.environ.get(key, 'not set')}")

print()
run_command("bash -lc 'type module >/dev/null 2>&1 && module --version 2>&1 | head -n 2 || echo MODULE_COMMAND=NOT_AVAILABLE'")

### Interpret what you observed

Answer these from the actual output:

1. How many Slurm partitions are visible?
2. Is there a GPU partition?
3. How many nodes are visible?
4. Is your Jupyter session already managed by Slurm?
5. What evidence shows that this is a cluster rather than one standalone workstation?

### Connection to the historical evolution

Try to identify at least three historical ideas in the current environment:

- commodity-style compute nodes;
- multicore CPUs;
- heterogeneous CPU/GPU resources;
- scheduler-managed resource sharing;
- modular software environment.

<details>
<summary><strong>Show explanation</strong></summary>

A modern teaching cluster inherits ideas from several HPC eras at once.

It uses networked nodes like the cluster era, multicore CPUs from the multicore era, GPU resources from heterogeneous computing, and a scheduler plus software modules to manage the system as a shared service.

</details>

## 13 - What changed, and what stayed the same?

### What changed?

- Hardware scale increased dramatically.
- Parallelism became central.
- Systems became heterogeneous.
- Data movement became a first-class performance issue.
- Energy and cooling became architectural constraints.
- Software stacks became more complex.

### What stayed the same?

- We still need a useful answer.
- Time-to-solution still matters.
- Components must still be balanced.
- Software must match the machine.
- Performance must still be measured.

### Final question

A future processor becomes **10x faster at arithmetic**, but memory bandwidth, network performance and power limits barely improve.

What happens?

<details>
<summary><strong>Show explanation</strong></summary>

Many applications will **not** become 10x faster.

The unchanged memory, network and power limits become relatively more important.

This is the recurring pattern of HPC evolution:

> **Solve one bottleneck, and another becomes visible.**

</details>

## 14 - Exit ticket

Answer each in one or two sentences.

### 1. Why did parallelism become unavoidable?

Your answer:

### 2. Why did commodity clusters matter?

Your answer:

### 3. What does the memory wall mean?

Your answer:

### 4. Why is peak FLOPS not enough to choose a system for an application?

Your answer:

### 5. What does heterogeneous HPC mean?

Your answer:

<details>
<summary><strong>Show explanation</strong></summary>

- **Parallelism:** single-processor improvements alone could no longer provide enough performance growth.
- **Commodity clusters:** standard processors, networks and software made large systems more modular and cost-effective.
- **Memory wall:** processors can perform arithmetic faster than data can be supplied from memory.
- **Peak FLOPS:** application performance depends on the complete system and the workload bottleneck.
- **Heterogeneous HPC:** different types of computing resources are combined in one system.

</details>

## What you should leave with

- HPC evolved because the **limiting factor kept changing**.
- Performance shifted from mainly faster processors toward **parallelism, scale and specialization**.
- Vector/SIMD ideas exploit regular data parallelism.
- Distributed systems add communication and synchronization costs.
- Commodity clusters made large-scale HPC more modular and economically scalable.
- Multicore CPUs and GPUs made heterogeneous programming increasingly important.
- FLOPS and HPL are useful measurements, but **time-to-solution and workload fit matter more for a real application**.
- Modern HPC is an ecosystem of compute, accelerators, memory, networks, storage and software.
- The current SciTech cluster contains ideas inherited from several generations of HPC.

Next: **M1.S3 - Architecture of HPC systems**.